# V8 — Hospital-Level Model Comparison (PyMC)

Convergence test for 3 nested hospital-level models, compared via **WAIC** and **holdout MSE**.

| Model | Mean function (log scale) |
|-------|---------------------------|
| M1 | $\alpha_j + \psi \cdot \text{lag}$ |
| M2 | $\alpha_j + \beta_j \cos(2\pi t/52) + \gamma_j \sin(2\pi t/52) + \psi \cdot \text{lag}$ |
| M3 | M2 + $\delta^{\text{NY}}_{r(j)}$ spike indicators |

where $\text{lag} = \log(y_{j,t-1}+1) - \text{base}_{j,t-1}$ is the observation-driven AR(1) correction.

**Shared structure:**
- Negative binomial likelihood on raw weekly trolley counts
- Partial pooling of hospital intercepts within HSE regions (non-centered)
- **70/30 temporal split**: train on first 119 weeks, recursive forecast on remaining 51

**WAIC** (Watanabe-Akaike IC) is the Bayesian analogue of AIC — both penalise complexity, but WAIC uses the full posterior rather than a point estimate. Lower = better.

In [ ]:
import json
import os
from pathlib import Path

os.environ.setdefault("PYTENSOR_FLAGS", "base_compiledir=/tmp/pytensor")
Path("/tmp/pytensor").mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings("ignore")

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from IPython.display import display

print(f"PyMC {pm.__version__}")

In [ ]:
# ── Find repo root ──
ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / 'data' / 'models').exists():
        ROOT = c
        break
DATA = ROOT / 'data'

# ── Hospital classification ──
with open(DATA / 'hospital_model_classification.json') as f:
    hosp_class = json.load(f)['hospitals']

# ── Load raw weekly data (full weeks only) ──
raw = pd.read_csv(
    DATA / 'raw data' / 'weekly_by_hospital_202604090859.csv',
    parse_dates=['week_start'],
)
raw = raw[raw['days_in_week'] == 7].copy()
raw['week_end'] = raw['week_start'] + pd.Timedelta(days=6)

# ── Drop all-zero hospitals ──
totals = raw.groupby('hospital')['total_trolleys'].sum()
all_zero = sorted(totals[totals == 0].index)

dropped = pd.DataFrame([
    {
        'hospital': h,
        'HSE model': hosp_class.get(h, {}).get('model', '?'),
        'note': hosp_class.get(h, {}).get('note', ''),
    }
    for h in all_zero
])
print(f"Dropped {len(all_zero)}/{raw['hospital'].nunique()} hospitals "
      f"({len(all_zero)/raw['hospital'].nunique()*100:.0f}%) — all-zero trolley counts:\n")
display(dropped)

# ── Pivot to (hospital x week) matrix ──
keep = raw[~raw['hospital'].isin(all_zero)]
pivot = (
    keep.pivot_table(index='hospital', columns='week_end',
                     values='total_trolleys', fill_value=0)
    .sort_index()
)
hospitals = list(pivot.index)
weeks = list(pivot.columns)
y_full = pivot.values.astype(int)
J, T_total = y_full.shape

# ── Region mapping ──
hosp_region = keep.drop_duplicates('hospital').set_index('hospital')['region']
regions_list = sorted(hosp_region.unique())
region_lookup = {r: i for i, r in enumerate(regions_list)}
region_idx = np.array([region_lookup[hosp_region[h]] for h in hospitals], dtype='int32')

# ── 70 / 30 temporal split ──
T_train = int(T_total * 0.7)
T_test = T_total - T_train
y_train = y_full[:, :T_train]
y_test = y_full[:, T_train:]

print(f"\n{J} hospitals, {len(regions_list)} regions, {T_total} total weeks")
print(f"Train: {T_train} weeks  ({pd.Timestamp(weeks[0]).date()} to "
      f"{pd.Timestamp(weeks[T_train - 1]).date()})")
print(f"Test:  {T_test} weeks  ({pd.Timestamp(weeks[T_train]).date()} to "
      f"{pd.Timestamp(weeks[-1]).date()})")

In [ ]:
# ── Event indicators (full series, matching JAGS data prep) ──
t_idx = np.arange(1, T_total + 1)
cos_t = np.cos(2 * np.pi * t_idx / 52)
sin_t = np.sin(2 * np.pi * t_idx / 52)
wk = t_idx % 52
ny_pre = (wk == 0).astype(float)
ny_mid = (wk == 1).astype(float)
ny_post = (wk == 2).astype(float)

# Training slices
cos_tr = cos_t[:T_train]
sin_tr = sin_t[:T_train]
ny_pre_tr = ny_pre[:T_train]
ny_mid_tr = ny_mid[:T_train]
ny_post_tr = ny_post[:T_train]
log_y_lag = np.log(y_train[:, :-1] + 1.0)  # (J, T_train-1)

# ── Model specifications ──
MODEL_SPECS = [
    {'name': 'M1', 'label': 'M1: alpha + AR(1)',                    'season': False, 'spikes': False},
    {'name': 'M2', 'label': 'M2: alpha + season + AR(1)',           'season': True,  'spikes': False},
    {'name': 'M3', 'label': 'M3: alpha + season + spikes + AR(1)',  'season': True,  'spikes': True},
]

coords = {'hospital': hospitals, 'region': regions_list,
          'week': np.arange(1, T_train + 1)}


def build_model(spec):
    with pm.Model(coords=coords) as model:
        # Hierarchical intercepts (non-centered)
        mu_global = pm.Normal('mu_global', mu=4.0, sigma=2.0)
        sigma_region = pm.HalfNormal('sigma_region', sigma=1.5)
        sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=1.5)
        mu_region_raw = pm.Normal('mu_region_raw', dims='region')
        alpha_raw = pm.Normal('alpha_raw', dims='hospital')
        mu_region = pm.Deterministic(
            'mu_region', mu_global + sigma_region * mu_region_raw, dims='region')
        alpha = pm.Deterministic(
            'alpha', mu_region[region_idx] + sigma_alpha * alpha_raw, dims='hospital')

        # Base mean on log scale: (J, T_train)
        base = alpha[:, None] + pt.zeros(T_train)[None, :]
        if spec['season']:
            beta = pm.Normal('beta', sigma=0.5, dims='hospital')
            gamma = pm.Normal('gamma', sigma=0.5, dims='hospital')
            base = base + beta[:, None] * cos_tr[None, :] + gamma[:, None] * sin_tr[None, :]
        if spec['spikes']:
            d_pre = pm.Normal('delta_pre', sigma=0.5, dims='region')
            d_mid = pm.Normal('delta_mid', sigma=0.5, dims='region')
            d_post = pm.Normal('delta_post', sigma=0.5, dims='region')
            base = base + d_pre[region_idx][:, None] * ny_pre_tr[None, :]
            base = base + d_mid[region_idx][:, None] * ny_mid_tr[None, :]
            base = base + d_post[region_idx][:, None] * ny_post_tr[None, :]

        psi = pm.Uniform('psi', lower=-1, upper=1)
        r = pm.Gamma('r', alpha=2, beta=0.5, dims='hospital')

        # Observation-driven AR(1) — vectorised, no scan
        log_mu_t1 = base[:, :1]                                                      # (J, 1)
        log_mu_rest = base[:, 1:] + psi * (pt.as_tensor(log_y_lag) - base[:, :-1])   # (J, T-1)
        log_mu = pt.concatenate([log_mu_t1, log_mu_rest], axis=1)

        pm.NegativeBinomial(
            'y_obs', mu=pt.exp(log_mu), alpha=r[:, None],
            observed=y_train, dims=('hospital', 'week'),
        )
    return model

In [ ]:
# ── Fit all models ──
results = {}
for spec in MODEL_SPECS:
    print(f"\n{'=' * 60}")
    print(f"Fitting {spec['label']}...")
    model = build_model(spec)
    with model:
        trace = pm.sample(
            draws=500, tune=500, chains=2, cores=2,
            target_accept=0.9, random_seed=42,
            return_inferencedata=True,
            idata_kwargs={'log_likelihood': True},
        )
    # Quick convergence check
    divs = int(trace.sample_stats['diverging'].sum())
    rhat = az.rhat(trace)
    max_rhat = max(float(rhat[v].max()) for v in rhat.data_vars)
    print(f"  Divergences: {divs}  |  Max R-hat: {max_rhat:.3f}")
    results[spec['name']] = {'spec': spec, 'trace': trace}

In [ ]:
# ── Recursive holdout forecast (mirrors LectureSet13 scheme) ──
#
# Step 1: y_hat uses last training observation
# Step m: y_hat uses previous forecast (not true value)

def holdout_mse(trace, spec):
    post = trace.posterior
    alpha_m = post['alpha'].mean(dim=['chain', 'draw']).values
    psi_m = float(post['psi'].mean())

    # Base for all weeks
    base_all = np.tile(alpha_m[:, None], (1, T_total))
    if spec['season']:
        base_all += post['beta'].mean(dim=['chain', 'draw']).values[:, None] * cos_t[None, :]
        base_all += post['gamma'].mean(dim=['chain', 'draw']).values[:, None] * sin_t[None, :]
    if spec['spikes']:
        dp = post['delta_pre'].mean(dim=['chain', 'draw']).values
        dm = post['delta_mid'].mean(dim=['chain', 'draw']).values
        dpo = post['delta_post'].mean(dim=['chain', 'draw']).values
        base_all += dp[region_idx][:, None] * ny_pre[None, :]
        base_all += dm[region_idx][:, None] * ny_mid[None, :]
        base_all += dpo[region_idx][:, None] * ny_post[None, :]

    # Recursive forecast on held-out weeks
    base_test = base_all[:, T_train:]
    y_prev = y_train[:, -1].astype(float)
    base_prev = base_all[:, T_train - 1]

    mu_hat = np.zeros((J, T_test))
    for t in range(T_test):
        log_mu_t = base_test[:, t] + psi_m * (np.log(y_prev + 1) - base_prev)
        mu_hat[:, t] = np.exp(log_mu_t)
        y_prev = mu_hat[:, t]
        base_prev = base_test[:, t]

    return float(np.mean((y_test - mu_hat) ** 2))


# ── Build comparison table ──
rows = []
for name, entry in results.items():
    spec = entry['spec']
    trace = entry['trace']
    waic = az.waic(trace)
    rhat = az.rhat(trace)
    max_rhat = max(float(rhat[v].max()) for v in rhat.data_vars)
    mse_val = holdout_mse(trace, spec)
    rows.append({
        'Model': spec['label'],
        'WAIC': round(float(-2 * waic.elpd_waic), 1),
        'p_WAIC': round(float(waic.p_waic), 1),
        'Holdout MSE': round(mse_val, 1),
        'Max R-hat': round(max_rhat, 3),
        'Divergences': int(trace.sample_stats['diverging'].sum()),
    })

comparison = pd.DataFrame(rows).sort_values('WAIC').reset_index(drop=True)
display(comparison)

# ── Bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), layout='constrained')
for ax, col, title in zip(axes, ['WAIC', 'Holdout MSE'],
                          ['WAIC (lower = better)', 'Holdout MSE (lower = better)']):
    ax.barh(comparison['Model'], comparison[col], color='#4c78a8')
    ax.invert_yaxis()
    ax.set_xlabel(col)
    ax.set_title(title)
plt.show()